
# QGCNN Graph-Batch Training

This notebook converts each dataset chunk into a k-NN graph batch, stores the batch graph on disk, and trains the quantum classifier batch by batch with checkpoint resume support.


In [ ]:

!pip install pandas numpy scikit-learn matplotlib seaborn networkx joblib dill qiskit==1.1.0 qiskit-machine-learning==0.8.0 qiskit-algorithms==0.3.1 qiskit-aer


In [ ]:

from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import seaborn as sns
from sklearn.metrics import (
    auc,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

from qcnn_resource_training import (
    collect_predictions_from_batches,
    create_qgcnn,
    load_graph_batch,
    prepare_graph_batches,
    resolve_data_path,
    score_qgcnn_on_batches,
    train_qgcnn_in_batches,
)


In [ ]:

DATA_PATH = resolve_data_path("creditcard_2023.csv", "creditcard_2023.csv.zip")
CACHE_DIR = Path("graph_cache") / DATA_PATH.stem.replace(".csv", "")
CHECKPOINT_DIR = Path("graph_checkpoints") / DATA_PATH.stem.replace(".csv", "")

prepared = prepare_graph_batches(
    DATA_PATH,
    cache_dir=CACHE_DIR,
    n_components=4,
    chunksize=25_000,
    test_size=0.2,
    random_state=42,
    k_neighbors=8,
    force_rebuild=False,
)

print(f"Dataset: {DATA_PATH.name}")
print(f"Graph cache: {prepared.cache_dir}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")
print(f"Train graph batches: {len(prepared.train_batch_files)}")
print(f"Test graph batches: {len(prepared.test_batch_files)}")
print(f"Training samples: {prepared.train_size:,}")
print(f"Evaluation samples: {prepared.test_size:,}")
print(f"Base feature columns: {len(prepared.feature_columns)}")
print(f"Graph feature width before PCA: {prepared.graph_feature_dim}")
print(f"Quantum input width after PCA: {prepared.n_components}")


In [ ]:

first_batch = load_graph_batch(prepared.train_batch_files[0])
print(f"First train graph batch: {first_batch.batch_file.name}")
print(f"Nodes: {first_batch.num_nodes:,}")
print(f"Edges: {first_batch.edge_index.shape[1]:,}")
print(f"Reduced feature shape: {first_batch.X.shape}")

preview_nodes = min(64, first_batch.num_nodes)
preview_graph = nx.Graph()
preview_graph.add_nodes_from(range(preview_nodes))

mask = (
    (first_batch.edge_index[0] < preview_nodes)
    & (first_batch.edge_index[1] < preview_nodes)
)
preview_edges = first_batch.edge_index[:, mask].T.tolist()
preview_graph.add_edges_from((int(src), int(dst)) for src, dst in preview_edges)

plt.figure(figsize=(7, 7))
pos = nx.spring_layout(preview_graph, seed=42)
nx.draw_networkx(
    preview_graph,
    pos=pos,
    node_size=80,
    with_labels=False,
    edge_color="#7aa6c2",
    node_color="#f3b562",
    alpha=0.85,
)
plt.title("Preview of One Graph Batch")
plt.axis("off")
plt.show()


In [ ]:

qc, input_params, weight_params = create_qgcnn(num_qubits=4)
print(qc)


In [ ]:

EPOCHS = 2
MAXITER_PER_BATCH = 6
EVAL_MAX_SAMPLES = 20_000

classifier, history = train_qgcnn_in_batches(
    prepared,
    qc,
    input_params,
    weight_params,
    checkpoint_dir=CHECKPOINT_DIR,
    epochs=EPOCHS,
    maxiter_per_batch=MAXITER_PER_BATCH,
    random_state=42,
    resume=True,
    eval_max_samples=EVAL_MAX_SAMPLES,
)

print(f"History entries: {len(history)}")
print("Rerun this cell to continue from the latest graph-batch checkpoint.")


In [ ]:

full_test_acc = score_qgcnn_on_batches(classifier, prepared.test_batch_files)
print(f"Full test accuracy across graph batches: {full_test_acc:.4f}")

PLOT_MAX_SAMPLES = 50_000
y_true, score_probs = collect_predictions_from_batches(
    classifier,
    prepared.test_batch_files,
    max_samples=PLOT_MAX_SAMPLES,
)
y_pred = (score_probs >= 0.5).astype(np.int8)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:\n")
print(cm)

auc_score = roc_auc_score(y_true, score_probs)
print(f"\nROC AUC Score: {auc_score:.4f}")

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Not Fraud", "Fraud"],
    yticklabels=["Not Fraud", "Fraud"],
    cbar=True,
    square=True,
)
plt.title("QGCNN Confusion Matrix", fontsize=14)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.tight_layout()
plt.show()

fpr, tpr, _ = roc_curve(y_true, score_probs)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc_score:.3f}")
plt.plot([0, 1], [0, 1], "k--", linewidth=1)
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("QGCNN ROC Curve", fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

precision, recall, _ = precision_recall_curve(y_true, score_probs)
pr_auc = auc(recall, precision)
print(f"PR AUC Score: {pr_auc:.4f}")

plt.figure(figsize=(6, 5))
plt.plot(recall, precision, linewidth=2, label=f"PR AUC = {pr_auc:.3f}")
plt.xlabel("Recall", fontsize=12)
plt.ylabel("Precision", fontsize=12)
plt.title("QGCNN Precision-Recall Curve", fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
